# 00 — Environment & Free/Low-Cost Providers

## Learning requirements
Sau notebook này bạn phải:
- tạo được Python virtual environment và Jupyter kernel;
- hiểu provider abstraction của LangChain;
- chạy cùng một prompt với Gemini/Groq/OpenRouter/Ollama mà không đổi business logic;
- biết **ngrok không phải LLM provider**;
- biết cách tránh phụ thuộc vào một model ID cố định.

## Provider strategy
1. **Gemini Developer API**: default của course.
2. **Groq**: provider thứ hai để test tốc độ và tool calling.
3. **OpenRouter**: fallback bằng `openrouter/free`/`:free`.
4. **Ollama**: local/offline, không mất API fee.
5. **ngrok**: chỉ dùng ở module MCP/deployment nếu cần expose local service.

> Free-tier quota/model availability là external constraint và có thể đổi. Vì vậy code luôn đọc provider/model từ `.env`.

In [ ]:
# Chạy một lần nếu kernel chưa có dependencies.
# %pip install -r ../../requirements.txt

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Notebook nằm sâu 2 cấp dưới root.
root = Path.cwd().resolve()
while root.name != "Langchain_Advanced" and root.parent != root:
    if (root / "requirements.txt").exists():
        break
    root = root.parent

load_dotenv(root / ".env")
print("Selected provider:", os.getenv("LLM_PROVIDER", "gemini"))

## API key checklist

Không bao giờ commit API key vào notebook/repository.

`.env` tối thiểu:

```env
LLM_PROVIDER=gemini
GOOGLE_API_KEY=...
```

Để đổi provider:

```env
LLM_PROVIDER=groq
GROQ_API_KEY=...
```

Model name cũng được cấu hình trong `.env.example`.

In [ ]:
import sys
sys.path.insert(0, str(root))

from src.providers import get_chat_model

model = get_chat_model()
response = model.invoke("Reply exactly with: provider setup works")
print(response.text if hasattr(response, "text") else response.content)

In [ ]:
# Provider portability test.
# Chỉ chạy provider nào bạn đã có credentials/local model.
for provider in ["gemini"]:  # TODO: thêm "groq", "openrouter", "ollama"
    m = get_chat_model(provider)
    r = m.invoke("Return one sentence explaining what an LLM provider is.")
    print(provider, "=>", getattr(r, "text", r.content))

## Required output

1. `.env` local chạy được với ít nhất **một** provider.
2. Chạy thành công portability test với ít nhất **hai** provider trước khi kết thúc Level 2.
3. Viết 5–10 dòng trong notebook giải thích:
   - provider abstraction giải quyết vấn đề gì;
   - model capability khác provider abstraction như thế nào;
   - tại sao free-tier không nên được xem là production SLA.

## Done criteria
- Không có secret trong Git.
- Bạn đổi provider chỉ bằng env/config.
- Bạn giải thích được Groq ≠ Grok và ngrok ≠ Groq.